In [ ]:
import numpy as np
import torch
from typing import Any, Callable, Optional, Sequence
from tqdm import tqdm
from torch.nn import BCEWithLogitsLoss
import torch.nn as nn
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall


In [ ]:
def setup_dataloader(dataset_type: str, 
                     save_to: str,seq_len: int, 
                     c_classes: int| list[int],
                     n_samples: int,
                     task_type: str,
                     spatial_factor: float,
                     time_factor: float,
                     time_window: float,
                     count_thr: int,
                     duration_ratio: float,
                     ctx_f_max: float,
                     split_percent: float,
                     batch_size: int,
                     dataloader_nb_workers: int = 1,
                     device: Optional[torch.device] = None,
                     ):
    
    # select a specific data 
    # create the dataset at a given location
    # split on multiple splits
    # create required transform for the data and context
    # create the dataloader
    if dataset_type.lower() == "dvsgesture":
        dataset_class = DVSGestureContext
    elif dataset_type.lower() == "shd":
        dataset_class = SDHContext
    else:
        raise ModuleNotFoundError(f"dataset {dataset_type.lower()} doesn't exist."\
                                  "Available dataset are [dvsgesture, shd]")
    sensor_size = dataset_class.sensor_size
    print(sensor_size)
    sensor_size = (
        int(math.ceil(sensor_size[0] * spatial_factor)),
        int(math.ceil(sensor_size[1] * spatial_factor)),
        sensor_size[2]
        )
    
    input_transform = tonic.transforms.Compose([
        TakeEventByTime(duration_ratio),
        tonic.transforms.Downsample(
            time_factor=time_factor,
            spatial_factor=spatial_factor
        ),
        BinarizeFrame(
            sensor_size=sensor_size,
            time_window=time_window,
            count_thr=count_thr),
        Flatten(),
    ])
    if isinstance(c_classes, int):
        c_classes = list(range(c_classes))
        
    context_transform = PoissonContextTransform(c_classes=c_classes, f_max=ctx_f_max)

    train_dataset = dataset_class(
        save_to=save_to,
        seq_len=seq_len,
        c_classes=c_classes,
        n_samples=n_samples,
        train=True,
        task_type=task_type,
        transform=input_transform,
        context_transform=context_transform
    )
    test_dataset = dataset_class(
        save_to=save_to,
        seq_len=seq_len,
        c_classes=c_classes,
        n_samples=n_samples,
        train=False,
        task_type=task_type,
        transform=input_transform,
        context_transform=context_transform
    )
    input_size = math.prod(sensor_size)
    
    # class_weighting for unbalanced task
    if task_type == "recognition":
        class_weights = torch.full(
             size=(), fill_value=max(1, seq_len - 1))
    else:
        weight = (3*seq_len - 1)/(seq_len + 1)
        class_weights = torch.full(
            (), fill_value=weight)
    
    cache_path = f"{save_to}/cache/{dataset_type.lower()}/" \
        f'{seq_len}_{"-".join(str(x) for x in train_dataset.c_classes)}' \
        f'_{spatial_factor}_{time_factor}'\
        f'_{time_window}_{count_thr}'\
        f'_{duration_ratio}_{task_type}_{ctx_f_max}'
    
    train_dataset, val_dataset = split_dataset(
        train_dataset, split_percent, shuffle_before_split=True)
    
    collate_fn = PadTensors(batch_first=True)
    train_dataset = DiskCachedDataset(
        train_dataset, contextual=True,
        cache_path=cache_path + "/train",
    )
    val_dataset = DiskCachedDataset(
        val_dataset, contextual=True,
        cache_path=cache_path + "/val",
    )
    test_dataset = DiskCachedDataset(
        test_dataset, contextual=True,
        cache_path=cache_path + "/test",
    )
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=collate_fn,
        num_workers=dataloader_nb_workers,
        persistent_workers=True,
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=collate_fn,
        num_workers=dataloader_nb_workers,
        persistent_workers=True,
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=collate_fn,
        num_workers=dataloader_nb_workers,
        persistent_workers=True,
    )
    return train_dataloader, val_dataloader, test_dataloader, input_size, class_weights

In [ ]:
def setup_dataset_and_model(config: argparse.Namespace):
    # namespace to dict
    config: dict = vars(config)
    seed = config["seed"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    np.random.seed(seed)
    torch.manual_seed(seed)
    print(device)
    # Dataset
    thr = 0.05
    if config["dataset"] == "dvsgesture":
        c_classes = 10
        nb_samples = 2048
        spatial_factor = 0.12
        time_factor = 1e-3
        time_window = 5.0
        count_thr = 3
        duration_ratio = 0.4
        rate_soma = 43
        rate_apical = 20
        
    elif config["dataset"] == "shd":
        c_classes = 20
        nb_samples = 4096
        spatial_factor = 0.5
        time_factor = 1e-3
        time_window = 2
        count_thr = 1
        duration_ratio = 1.0
        rate_soma = 50
        rate_apical = 20
        
    train_dl, val_dl, test_dl, input_size, class_weight = setup_dataloader(
        config["dataset"], config["data_path"], 
        config["seq_len"], c_classes, nb_samples, 
        config["task_type"], spatial_factor, time_factor, time_window, count_thr,
        duration_ratio, ctx_f_max=200, split_percent=0.8, batch_size=config["batch_size"],
        dataloader_nb_workers=config["nb_worker"], device=device)


        
    return (train_dl, val_dl, test_dl), config, device, class_weight

In [ ]:
dataloaders, config, device, class_weight = setup_dataset_and_model(parser.parse_args())
train_dataloader, val_dataloader, test_dataloader = dataloaders


steps=300
lr=0.005

input_size, tau_beta = 2, 5, 50.0


model = Simple_RSNN(
    input_size=input_size,
    hidden_size=128,
    output_size=2,
    tau_soma=20.0,
    tau_beta=tau_beta,
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

loss_fn = BCEWithLogitsLoss(
    reduction='mean',
    pos_weight=class_weight)
    # reg_loss = self.l2*regularization_multi_layers(
    #     states, 1e-3 * inputs[0].size(1), self.target_rate, 0, "upper")

In [ ]:





acc = BinaryAccuracy().to(device)
precision = BinaryPrecision().to(device)
recall = BinaryRecall().to(device)
# in the memorization task we train on the full sequence, but evaluate on the last item
# this is because we want to evaluate the memorization capabilities.
# the last item have equiprobable pos/neg target probability.
eval_at_n = -1
if config["task_type"] == "memory":
    eval_at_n = config["seq_len"] - 1
for i in range(config["epoch"]):
    epoch_loss = 0
    for batch in tqdm(train_dataloader, "Training epoch"):
        optimizer.zero_grad()
        x, c, targets, block_idx = batch


        x = x.to(device)
        c = c.to(device)
        targets = targets.to(device)
        block_idx = block_idx.to(device)
        time_steps = (block_idx < config["seq_len"]).sum(1)
        outputs = model((x, c))
        spike_sum = model.get_spike_sums()
        
        reg_loss, spike_proba = layer_wise_regularization(spike_sum[0], config["target_spike_prob"], time_steps)
        outputs_reduce, targets_reduce = by_block_summation(
            outputs, targets, block_idx)
        acc(outputs_reduce, targets_reduce)
        precision(outputs_reduce, targets_reduce)
        recall(outputs_reduce, targets_reduce)
        loss = loss_fn(outputs_reduce, targets_reduce.float())
        loss += config["spike_reg_l2"] * reg_loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss
    lr_scheduler.step()
    train_epoch_loss =  epoch_loss / len(train_dataloader) 
    train_acc = acc.compute()
    train_precision = precision.compute()
    train_recall = recall.compute()
    acc.reset(), precision.reset(), recall.reset()
    epoch_loss = 0
    if val_dataloader is not None:
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation epoch:"):
                x, c, targets, block_idx = batch
                x = x.to(device)
                c = c.to(device)
                targets = targets.to(device)
                block_idx = block_idx.to(device)
                outputs = model((x, c))
                outputs_reduce, targets_reduce = by_block_summation(
                    outputs, targets, block_idx)
                if eval_at_n != -1:
                    outputs_reduce = outputs_reduce[:, eval_at_n]
                    targets_reduce = targets_reduce[:, eval_at_n]
                acc(outputs_reduce, targets_reduce)
                precision(outputs_reduce, targets_reduce)
                recall(outputs_reduce, targets_reduce)
                loss = loss_fn(outputs_reduce, targets_reduce.float())
                epoch_loss += loss
        with torch.no_grad():
            val_epoch_loss =  epoch_loss / len(val_dataloader) 
            val_acc = acc.compute()
            val_precision = precision.compute()
            val_recall = recall.compute()
            str_format = "Epoch n-{} train_loss: {}, train_acc: {}, train_precision: {}, train_recall {}\n"\
                        "           vall_loss: {}, val_acc: {}, val_precision: {}, val_recall {}"
                
            print(str_format.format(i+1, train_epoch_loss, train_acc, train_precision, train_recall,
                                    val_epoch_loss, val_acc, val_precision, val_recall))
            acc.reset(), precision.reset(), recall.reset()
            epoch_loss = 0
if test_dataloader is not None:
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing epoch:"):
            x, c, targets, block_idx = batch
            x = x.to(device)
            c = c.to(device)
            targets = targets.to(device)
            block_idx = block_idx.to(device)
            outputs = model((x, c))
            outputs_reduce, targets_reduce = by_block_summation(
                outputs, targets, block_idx)
            if eval_at_n != -1:
                outputs_reduce = outputs_reduce[:, eval_at_n]
                targets_reduce = targets_reduce[:, eval_at_n]
            acc(outputs_reduce, targets_reduce)
            precision(outputs_reduce, targets_reduce)
            recall(outputs_reduce, targets_reduce)
            loss = loss_fn(outputs_reduce, targets_reduce.float())
            epoch_loss += loss
        test_epoch_loss =  epoch_loss / len(test_dataloader) 
        test_acc = acc.compute()
        test_precision = precision.compute()
        test_recall = recall.compute()
    str_format = "Testing: loss: {}, accuracy: {}, precision: {}, recall {}"
    print(str_format.format(test_epoch_loss, test_acc, test_precision, test_recall))